# Push-Purchase Relationship Analysis

This notebook examines the relationship between push notifications and purchase behavior, exploring how pushes affect buying patterns.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import glob
import os
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

In [2]:
# Get all push files
push_files = glob.glob("../../data/data1031/sleep_push_result_*.csv")

# Read and combine all push data
all_push_data_list = []
for file in push_files:
    df = pd.read_csv(file)
    df = df[['dt', 'member_id', 'trigger_tag', 'proposal_name', 'coupon', 'discount', 'policy_id']]
    all_push_data_list.append(df)

all_push_data = pd.concat(all_push_data_list, ignore_index=True)
all_push_data['dt'] = pd.to_datetime(all_push_data['dt'])
all_push_data.head()

folder_path = '../../data/data1031'
file = 'order_result.csv'
file_path = os.path.join(folder_path, file)
columns = ["member_id", "create_hour", 
            "coffee_origin_money", "drink_not_coffee_origin_money",
            "coffee_commodity_num", "coffee_top_commodity_num",
            "not_coffee_commodity_num", "not_coffee_top_commodity_num",
            "food_commodity_num",
            "use_coupon_num",
            "coffee_discount", "disount_tag",
            "drink_not_coffee_discount"]
# only read the columns in the list
order_result = pd.read_csv(file_path, usecols=columns)
order_result['dt'] = pd.to_datetime(order_result['create_hour'],
                                             format='%Y-%m-%d %H').dt.date
order_result['total_items'] = (order_result['coffee_commodity_num'] + 
                               order_result['not_coffee_commodity_num'])
order_result['total_top_items'] = (order_result['coffee_top_commodity_num'] + 
                                   order_result['not_coffee_top_commodity_num'])                                   
order_result['has_discount'] = (~order_result['coffee_discount'].isnull()) | (~order_result['drink_not_coffee_discount'].isnull())
order_result['drink_not_coffee_commodity_num'] = order_result['not_coffee_commodity_num'] - order_result['food_commodity_num']
# discard column 'create_hour'
order_result = order_result.drop(columns=['create_hour'])
order_result.head()

,dt,member_id,trigger_tag,proposal_name,coupon,discount,policy_id
0,2021-02-24,86732,1,全场饮品抵用金40元券包,40.0,NaN,556f73c469a4cad3ba1da84d62035a2e
1,2020-08-23,613008,2,NaN,NaN,NaN,6e39b52e3b0700c2684b8eb0d2fd75e3
2,2021-06-14,613008,4,NaN,NaN,NaN,3349bc61a123584ee53025936922321b
3,2021-12-07,613008,1,4.2折饮品券,NaN,4.2,76b95b3734c97e72c17b20bcc63a4a75
4,2021-09-13,613008,4,NaN,NaN,NaN,e7944450abe0142c725b227605c17bfd


,member_id,coffee_commodity_num,coffee_top_commodity_num,not_coffee_commodity_num,not_coffee_top_commodity_num,food_commodity_num,use_coupon_num,coffee_origin_money,coffee_discount,drink_not_coffee_origin_money,drink_not_coffee_discount,disount_tag,dt,total_items,total_top_items,has_discount,drink_not_coffee_commodity_num
0,1,2,0,0,0,0,1,61.0,0.5,0.0,NaN,0,2021-05-29,2,0,True,0
1,1,1,1,2,0,2,1,29.0,0.4,0.0,NaN,0,2021-06-29,3,1,True,0
2,1,0,0,3,0,2,1,0.0,NaN,35.0,0.5,0,2021-06-29,3,0,True,1
3,3,3,2,0,0,0,0,90.0,0.6,0.0,NaN,3,2021-06-08,3,2,True,0
4,3,1,1,0,0,0,0,32.0,0.6,0.0,NaN,3,2021-11-13,1,1,True,0


In [ ]:
# 1. Merge push data and purchase data with proper indicators
# First, add source indicators to both datasets
all_push_data['data_source'] = 'push'
order_result['data_source'] = 'purchase'
order_result['dt'] = pd.to_datetime(order_result['dt'])

# Get all unique columns from both datasets
push_columns = set(all_push_data.columns)
purchase_columns = set(order_result.columns)
all_columns = push_columns.union(purchase_columns)

# Create full column sets for both datasets with NaN for missing columns
push_full = all_push_data.copy()
purchase_full = order_result.copy()

# Add missing columns to push data (purchase-specific columns)
for col in purchase_columns - push_columns:
    push_full[col] = np.nan

# Add missing columns to purchase data (push-specific columns)  
for col in push_columns - purchase_columns:
    purchase_full[col] = np.nan

# Combine both datasets
combined_data = pd.concat([push_full, purchase_full], ignore_index=True)

# Sort by member_id and datetime
combined_data = combined_data.sort_values(['member_id', 'dt']).reset_index(drop=True)

# 2. Add "new" column to identify first appearance of each member
combined_data['new'] = 0

# Mark the first appearance (earliest datetime) for each member as new = 1
first_appearances = combined_data.groupby('member_id')['dt'].idxmin()
combined_data.loc[first_appearances, 'new'] = 1

# Alternative simpler approach:
# first_appearances = combined_data.groupby('member_id').head(1).index
# combined_data.loc[first_appearances, 'new'] = 1

# 3. Calculate "dormant" status efficiently
def calculate_dormant_simple(df):
    """
    Simple and robust method to calculate dormant status
    """
    result_df = df.copy()
    
    # Initialize dormant column
    result_df['dormant'] = 0
    
    # Process each member separately
    for member_id in result_df['member_id'].unique():
        member_mask = result_df['member_id'] == member_id
        member_data = result_df[member_mask].copy().sort_values('dt')
        
        # Get all purchase dates for this member
        purchase_dates = member_data[member_data['data_source'] == 'purchase']['dt'].tolist()
        
        # For each row in this member's timeline
        for idx, row in member_data.iterrows():
            current_date = row['dt']
            is_new = row['new']
            
            # If this is the first appearance, not dormant
            if is_new == 1:
                result_df.loc[idx, 'dormant'] = 0
                continue
            
            # Find previous purchases (before current date)
            previous_purchases = [d for d in purchase_dates if d < current_date]
            
            if not previous_purchases:
                # No previous purchases but not new - treat as dormant
                result_df.loc[idx, 'dormant'] = 1
            else:
                # Find the most recent previous purchase
                last_purchase = max(previous_purchases)
                days_since_purchase = (current_date - last_purchase).days
                result_df.loc[idx, 'dormant'] = 1 if days_since_purchase > 30 else 0
    
    return result_df

# More efficient vectorized approach
def calculate_dormant_vectorized(df):
    """
    More efficient vectorized approach for large datasets
    """
    result_df = df.copy()
    
    # Create a helper dataframe with purchase dates per member
    purchase_dates_df = result_df[result_df['data_source'] == 'purchase'][['member_id', 'dt']].copy()
    purchase_dates_df = purchase_dates_df.rename(columns={'dt': 'purchase_date'})
    
    # Sort the main dataframe
    result_df = result_df.sort_values(['member_id', 'dt']).reset_index(drop=True)
    
    # Initialize dormant column
    result_df['dormant'] = 0
    
    # For each row, find the most recent purchase before current date
    for idx, row in result_df.iterrows():
        member_id = row['member_id']
        current_date = row['dt']
        
        # Skip if this is a new member
        if row['new'] == 1:
            continue
            
        # Get all purchases for this member before current date
        member_purchases = purchase_dates_df[
            (purchase_dates_df['member_id'] == member_id) & 
            (purchase_dates_df['purchase_date'] < current_date)
        ]
        
        if member_purchases.empty:
            # No previous purchases but not new - dormant
            result_df.loc[idx, 'dormant'] = 1
        else:
            # Find most recent purchase
            last_purchase = member_purchases['purchase_date'].max()
            days_since_purchase = (current_date - last_purchase).days
            if days_since_purchase > 30:
                result_df.loc[idx, 'dormant'] = 1
    
    return result_df

print("Calculating dormant status...")
final_data = calculate_dormant_vectorized(combined_data)

# Display basic information
print("Final dataset shape:", final_data.shape)
print("\nData types:")
print(final_data.dtypes)
print("\nFirst 10 rows:")
print(final_data[['member_id', 'dt', 'data_source', 'new', 'dormant']].head(10))

# Basic statistics
print("\n=== BASIC STATISTICS ===")
print("Total records:", len(final_data))
print("Unique members:", final_data['member_id'].nunique())
print("Push records:", len(final_data[final_data['data_source'] == 'push']))
print("Purchase records:", len(final_data[final_data['data_source'] == 'purchase']))

print("\n=== STATUS DISTRIBUTION ===")
print("New member status:")
print(final_data['new'].value_counts())
print("\nDormant status:")
print(final_data['dormant'].value_counts())

print("\n=== CROSS ANALYSIS ===")
print("Data source by New status:")
print(pd.crosstab(final_data['data_source'], final_data['new']))
print("\nData source by Dormant status:")
print(pd.crosstab(final_data['data_source'], final_data['dormant']))

# Validation
print("\n=== VALIDATION ===")
# Check that new=1 members have dormant=0
new_members_dormant = final_data[final_data['new'] == 1]['dormant'].value_counts()
print("Dormant status for new members (should all be 0):")
print(new_members_dormant)

# Check that each member has exactly one new=1
new_counts = final_data.groupby('member_id')['new'].sum()
print(f"\nMembers with exactly one 'new' mark: {(new_counts == 1).sum()}")
print(f"Members with no 'new' mark: {(new_counts == 0).sum()}")
print(f"Members with multiple 'new' marks: {(new_counts > 1).sum()}")

# Show sample member timelines
print(f"\n=== SAMPLE MEMBER TIMELINES ===")
sample_members = final_data['member_id'].unique()[:2]
for member_id in sample_members:
    print(f"\nMember {member_id}:")
    timeline = final_data[final_data['member_id'] == member_id][
        ['dt', 'data_source', 'new', 'dormant']
    ].sort_values('dt').reset_index(drop=True)
    print(timeline)
